# Notebook 06 — Feature Engineering
## Section 1: Build Modeling Features (Multi-Month)

Turn merged ICE + weather data into ML-ready columns for **regression on `delay_in_min`**.

**Feature groups**
- **Time** — from `departure_planned_time` (planned, not actual)
- **Operational** — station, train, route position
- **Weather** — temperature, precipitation, wind, etc.

**Rules**
- Target = `delay_in_min`
- Metric later = **MAE**
- **No leakage:** exclude `arrival_change_time`, `departure_change_time`
- Drop redundant `rain` (same as `precipitation`); drop `snowfall` if zero in summer

**Outputs**
- `ice_modeling_ready_YYYY-MM.parquet`
- `feature_manifest.json`

In [1]:
# =============================================================================
# Notebook 06 | Section 1: Feature Engineering (Multi-Month)
# =============================================================================
# Load ice_weather_merged from disk → time/operational/weather features.
# Keep categoricals as strings (encoding in Notebook 07). No target leakage.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PATH = REFERENCE_DIR / "project_config.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"

FEATURE_MANIFEST_PATH = REFERENCE_DIR / "feature_manifest.json"
FEATURE_REPORT_PATH = REFERENCE_DIR / "feature_engineering_report_multi_month.json"

PRIMARY_TARGET = "delay_in_min"
LEAKY_COLS = [
    "arrival_change_time",
    "departure_change_time",
    "delay_in_min",
]

WEATHER_CANDIDATES = [
    "temperature_2m",
    "precipitation",
    "rain",
    "snowfall",
    "windspeed_10m",
    "windgusts_10m",
    "weather_code",
    "visibility",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Time features from planned departure only (no leakage)."""
    out = df.copy()
    dep = pd.to_datetime(out["departure_planned_time"])

    out["dep_hour"] = dep.dt.hour.astype("int16")
    out["dep_day_of_week"] = dep.dt.dayofweek.astype("int8")  # Mon=0
    out["dep_month"] = dep.dt.month.astype("int8")
    out["is_weekend"] = (out["dep_day_of_week"] >= 5).astype("int8")
    out["is_rush_hour"] = out["dep_hour"].isin([7, 8, 9, 17, 18, 19]).astype("int8")
    return out


def select_weather_features(df: pd.DataFrame) -> list[str]:
    """Pick weather cols; drop rain (redundant) and snowfall if all zero."""
    available = [c for c in WEATHER_CANDIDATES if c in df.columns]
    selected = []

    for col in available:
        if col == "rain":
            continue  # redundant with precipitation
        if col == "snowfall":
            if df[col].fillna(0).abs().sum() == 0:
                continue
        selected.append(col)

    return selected


def build_features(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Return modeling dataframe + feature group metadata."""
    # Modeling rows: mergeable + weather matched (same as EDA)
    if "mergeable" in df.columns:
        work = df[df["mergeable"]].copy()
    else:
        work = df[df["departure_planned_hour_naive"].notna()].copy()

    if "weather_matched" in work.columns:
        work = work[work["weather_matched"]].copy()
    else:
        work = work[work["temperature_2m"].notna()].copy()

    work = add_time_features(work)
    weather_features = select_weather_features(work)

    # Simple derived weather flag
    if "precipitation" in weather_features:
        work["is_raining"] = (work["precipitation"] > 0).astype("int8")
        weather_derived = ["is_raining"]
    else:
        weather_derived = []

    time_features = [
        "dep_hour",
        "dep_day_of_week",
        "dep_month",
        "is_weekend",
        "is_rush_hour",
    ]
    operational_features = [
        "eva",
        "train_number",
        "train_line_station_num",
        "final_destination_station",
    ]
    operational_features = [c for c in operational_features if c in work.columns]

    # Keep IDs / timestamps for time-based split later (not used as features)
    meta_cols = [
        c
        for c in [
            "id",
            "month",
            "departure_planned_time",
            "departure_planned_hour_naive",
            "station_name",
            "xml_station_name",
        ]
        if c in work.columns
    ]

    feature_cols = time_features + operational_features + weather_features + weather_derived
    keep_cols = meta_cols + feature_cols + [PRIMARY_TARGET]

    # Drop duplicates while preserving order
    seen = set()
    keep_cols = [c for c in keep_cols if not (c in seen or seen.add(c))]

    missing_target = work[PRIMARY_TARGET].isna().sum()
    if missing_target > 0:
        raise ValueError(f"Target has {missing_target} nulls — check cleaning.")

    modeling_df = work[keep_cols].copy()

    info = {
        "rows_in": int(len(df)),
        "rows_out": int(len(modeling_df)),
        "time_features": time_features,
        "operational_features": operational_features,
        "weather_features": weather_features,
        "weather_derived": weather_derived,
        "all_feature_columns": feature_cols,
        "target": PRIMARY_TARGET,
        "leaky_excluded": LEAKY_COLS,
        "dropped_rain_reason": "redundant with precipitation",
    }
    return modeling_df, info


def process_month(target_month: str) -> dict:
    input_path = PROCESSED_DIR / f"ice_weather_merged_{target_month}.parquet"
    output_path = PROCESSED_DIR / f"ice_modeling_ready_{target_month}.parquet"

    if not input_path.exists():
        raise FileNotFoundError(f"Missing: {input_path}\nRun Notebook 04 first.")

    print("=" * 72)
    print(f"FEATURE ENGINEERING — {target_month}")
    print("=" * 72)
    print(f"Loading: {input_path.name}")

    df = pd.read_parquet(input_path)
    df["month"] = target_month

    modeling_df, info = build_features(df)
    modeling_df.to_parquet(output_path, index=False)

    print(f"Rows in merged file : {info['rows_in']:,}")
    print(f"Rows modeling-ready : {info['rows_out']:,}")
    print(f"Features            : {len(info['all_feature_columns'])}")
    print(f"Saved               : {output_path.name}")
    print()

    return {
        "month": target_month,
        "input_parquet": str(input_path),
        "output_parquet": str(output_path),
        **info,
    }


# =============================================================================
# RUN ALL MONTHS
# =============================================================================
config = load_json(CONFIG_PATH)
merge_strategy = load_json(MERGE_STRATEGY_PATH)
TARGET_MONTHS = config["scope"]["target_months"]

print("Notebook 06 | Section 1 — Feature engineering")
print(f"Months : {', '.join(TARGET_MONTHS)}")
print(f"Target : {PRIMARY_TARGET} | Metric later: MAE")
print()

month_results = [process_month(m) for m in TARGET_MONTHS]

# Combined file (optional but useful for NB07)
combined_frames = [
    pd.read_parquet(PROCESSED_DIR / f"ice_modeling_ready_{m}.parquet") for m in TARGET_MONTHS
]
combined_df = pd.concat(combined_frames, ignore_index=True)
combined_path = PROCESSED_DIR / "ice_modeling_ready_all_months.parquet"
combined_df.to_parquet(combined_path, index=False)

# Feature manifest (shared across months)
sample = month_results[0]
manifest = {
    "metadata": {
        "notebook": "06_Feature_Engineering",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "encoding_note": "Categorical columns kept as strings; encode in Notebook 07.",
    },
    "feature_groups": {
        "time_features": sample["time_features"],
        "operational_features": sample["operational_features"],
        "weather_features": sample["weather_features"],
        "weather_derived": sample["weather_derived"],
        "all_model_features": sample["all_feature_columns"],
    },
    "excluded_from_features": merge_strategy.get("columns_excluded_from_features", LEAKY_COLS),
    "leaky_columns_never_use": LEAKY_COLS,
    "output_files": {
        "per_month_pattern": "data/processed/ice_modeling_ready_YYYY-MM.parquet",
        "combined": str(combined_path),
        "months": [r["output_parquet"] for r in month_results],
    },
}

with FEATURE_MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(manifest), f, indent=2, ensure_ascii=False)

report = {
    "metadata": manifest["metadata"],
    "months": month_results,
    "totals": {
        "modeling_rows_all": int(sum(r["rows_out"] for r in month_results)),
        "combined_rows": int(len(combined_df)),
        "n_features": len(sample["all_feature_columns"]),
    },
    "manifest_path": str(FEATURE_MANIFEST_PATH),
}

with FEATURE_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE")
print(sep)
print(f"{'Month':<10} {'Modeling rows':>14} {'Features':>10}")
for r in month_results:
    print(
        f"{r['month']:<10} {r['rows_out']:>14,} "
        f"{len(r['all_feature_columns']):>10}"
    )
print()
print(f"Combined file : {combined_path}")
print(f"Manifest      : {FEATURE_MANIFEST_PATH}")
print(f"Report        : {FEATURE_REPORT_PATH}")
print()
print("Feature groups:")
print(f"  Time        : {sample['time_features']}")
print(f"  Operational : {sample['operational_features']}")
print(f"  Weather     : {sample['weather_features'] + sample['weather_derived']}")
print()
print("Next: Section 2 — validate features + prepare train/test split plan")
print(sep)

Notebook 06 | Section 1 — Feature engineering
Months : 2024-07, 2024-08, 2024-09
Target : delay_in_min | Metric later: MAE

FEATURE ENGINEERING — 2024-07
Loading: ice_weather_merged_2024-07.parquet
Rows in merged file : 146,389
Rows modeling-ready : 132,268
Features            : 16
Saved               : ice_modeling_ready_2024-07.parquet

FEATURE ENGINEERING — 2024-08
Loading: ice_weather_merged_2024-08.parquet
Rows in merged file : 136,321
Rows modeling-ready : 122,794
Features            : 16
Saved               : ice_modeling_ready_2024-08.parquet

FEATURE ENGINEERING — 2024-09
Loading: ice_weather_merged_2024-09.parquet
Rows in merged file : 135,547
Rows modeling-ready : 121,964
Features            : 16
Saved               : ice_modeling_ready_2024-09.parquet

Section 1 COMPLETE
Month       Modeling rows   Features
2024-07           132,268         16
2024-08           122,794         16
2024-09           121,964         16

Combined file : c:\Users\Manikanta\Desktop\ML Project\Not

# Notebook 06 — Feature Engineering
## Section 2: Validate Features & Time-Based Split

**Validate**
- All feature columns present
- No nulls in features/target
- No leakage columns used as inputs

**Split (time-based, no random shuffle)**
- **Train:** Jul + Aug 2024  
- **Test:** Sep 2024  

**Outputs**
- `ice_train.parquet` / `ice_test.parquet`
- `train_test_split_config.json`
- `feature_validation_report.json`

In [3]:
# =============================================================================
# Notebook 06 | Section 2: Validate Features & Time-Based Split
# =============================================================================
# Load ice_modeling_ready from disk → validate → drop unusable features
# (visibility is all-null in summer) → save train/test parquets.
# Time split: train=Jul+Aug, test=Sep (no random shuffle).
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MANIFEST_PATH = REFERENCE_DIR / "feature_manifest.json"
VALIDATION_REPORT = REFERENCE_DIR / "feature_validation_report.json"
SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

TRAIN_MONTHS = ["2024-07", "2024-08"]
TEST_MONTHS = ["2024-09"]
PRIMARY_TARGET = "delay_in_min"
LEAKY_COLS = ["arrival_change_time", "departure_change_time"]
MAX_FEATURE_NULL_PCT = 0.0  # drop any feature with nulls; visibility is 100% null


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


manifest = load_json(MANIFEST_PATH)
groups = manifest["feature_groups"]

ALL_FEATURES = list(groups["all_model_features"])
TIME_FEATURES = list(groups["time_features"])
OPERATIONAL_FEATURES = list(groups["operational_features"])
WEATHER_FEATURES = list(groups["weather_features"]) + list(groups.get("weather_derived", []))

print("Notebook 06 | Section 2 — Validate + time split")
print(f"Target : {PRIMARY_TARGET} | Metric: MAE")
print(f"Train  : {', '.join(TRAIN_MONTHS)}")
print(f"Test   : {', '.join(TEST_MONTHS)}")
print()

# =============================================================================
# 1. LOAD MODELING-READY DATA FROM DISK
# =============================================================================
frames = []
for month in TRAIN_MONTHS + TEST_MONTHS:
    path = PROCESSED_DIR / f"ice_modeling_ready_{month}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}\nRun Notebook 06 Section 1 first.")
    print(f"Loading: {path.name}")
    frames.append(pd.read_parquet(path))

df = pd.concat(frames, ignore_index=True)
print(f"Total rows loaded: {len(df):,}")
print()

# =============================================================================
# 2. FEATURE VALIDATION + DROP UNUSABLE COLUMNS
# =============================================================================
print("=" * 72)
print("FEATURE VALIDATION")
print("=" * 72)

missing_features = [c for c in ALL_FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing feature columns: {missing_features}")

leakage_in_features = [c for c in LEAKY_COLS if c in ALL_FEATURES]
if leakage_in_features:
    raise ValueError(f"Leakage columns in feature list: {leakage_in_features}")

null_profile = []
for col in ALL_FEATURES + [PRIMARY_TARGET]:
    n_null = int(df[col].isna().sum())
    null_pct = round(100 * n_null / len(df), 4)
    null_profile.append(
        {"column": col, "null_count": n_null, "null_pct": null_pct}
    )
    status = "OK" if n_null == 0 else "WARN"
    print(f"  [{status}] {col:<30} nulls={n_null:,} ({null_pct}%)")

# Drop features with nulls (visibility is 100% null in Jul–Sep Open-Meteo data)
dropped_features = [
    p["column"]
    for p in null_profile
    if p["column"] != PRIMARY_TARGET and p["null_pct"] > MAX_FEATURE_NULL_PCT
]

if dropped_features:
    print()
    print("Dropping unusable features:")
    for col in dropped_features:
        p = next(x for x in null_profile if x["column"] == col)
        print(f"  - {col}: {p['null_pct']}% null")
    ALL_FEATURES = [c for c in ALL_FEATURES if c not in dropped_features]
    WEATHER_FEATURES = [c for c in WEATHER_FEATURES if c not in dropped_features]

target_nulls = int(df[PRIMARY_TARGET].isna().sum())
if target_nulls > 0:
    raise ValueError(f"Target has {target_nulls} nulls — check cleaning.")

remaining_bad = [
    p for p in null_profile
    if p["column"] in ALL_FEATURES and p["null_count"] > 0
]
if remaining_bad:
    raise ValueError(f"Nulls still in features: {remaining_bad}")

print()
print(f"Features kept : {len(ALL_FEATURES)}")
print(f"  Time        : {len([c for c in TIME_FEATURES if c in ALL_FEATURES])}")
print(f"  Operational : {len([c for c in OPERATIONAL_FEATURES if c in ALL_FEATURES])}")
print(f"  Weather     : {len([c for c in WEATHER_FEATURES if c in ALL_FEATURES])}")
print("Validation: PASS")
print()

OPERATIONAL_ONLY = [c for c in TIME_FEATURES + OPERATIONAL_FEATURES if c in ALL_FEATURES]
FULL_FEATURES = ALL_FEATURES

# =============================================================================
# 3. TIME-BASED TRAIN / TEST SPLIT
# =============================================================================
train_df = df[df["month"].isin(TRAIN_MONTHS)].copy()
test_df = df[df["month"].isin(TEST_MONTHS)].copy()

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError("Train or test set is empty — check month column.")

overlap = set(train_df["id"]) & set(test_df["id"])
if overlap:
    raise ValueError(f"Train/test id overlap: {len(overlap)} rows")

train_df.to_parquet(TRAIN_PATH, index=False)
test_df.to_parquet(TEST_PATH, index=False)

print("=" * 72)
print("TIME-BASED SPLIT")
print("=" * 72)
print(f"Train rows : {len(train_df):,}  ({', '.join(TRAIN_MONTHS)})")
print(f"Test rows  : {len(test_df):,}  ({', '.join(TEST_MONTHS)})")
print()
print(f"Train target mean   : {train_df[PRIMARY_TARGET].mean():.2f} min")
print(f"Train target median : {train_df[PRIMARY_TARGET].median():.2f} min")
print(f"Test target mean    : {test_df[PRIMARY_TARGET].mean():.2f} min")
print(f"Test target median  : {test_df[PRIMARY_TARGET].median():.2f} min")
print()
print(f"Saved train: {TRAIN_PATH}")
print(f"Saved test : {TEST_PATH}")
print()

# =============================================================================
# 4. SAVE CONFIG + REPORTS (+ update manifest)
# =============================================================================
split_config = {
    "metadata": {
        "notebook": "06_Feature_Engineering",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "split_method": "time_based_by_month",
        "no_random_shuffle": True,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "split": {
        "train_months": TRAIN_MONTHS,
        "test_months": TEST_MONTHS,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "train_parquet": str(TRAIN_PATH),
        "test_parquet": str(TEST_PATH),
    },
    "dropped_features": dropped_features,
    "drop_reason": "visibility had 100% nulls in Jul–Sep Open-Meteo archive data",
    "feature_sets_for_ablation": {
        "operational_only": OPERATIONAL_ONLY,
        "full_with_weather": FULL_FEATURES,
        "note": "Notebook 07 trains both sets to answer RQ1/RQ2 with MAE.",
    },
    "columns": {
        "all_features": ALL_FEATURES,
        "target": PRIMARY_TARGET,
        "id_column": "id",
        "time_column_for_audit": "departure_planned_time",
    },
    "ready_for_notebook_07": True,
}

with SPLIT_CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(split_config), f, indent=2, ensure_ascii=False)

validation_report = {
    "metadata": split_config["metadata"],
    "validation_passed": True,
    "rows_total": int(len(df)),
    "null_profile": null_profile,
    "dropped_features": dropped_features,
    "features_kept": ALL_FEATURES,
    "leakage_check": {
        "leaky_columns_checked": LEAKY_COLS,
        "leakage_in_feature_list": leakage_in_features,
        "pass": len(leakage_in_features) == 0,
    },
    "split_summary": split_config["split"],
}

with VALIDATION_REPORT.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(validation_report), f, indent=2, ensure_ascii=False)

# Update manifest on disk so later notebooks use final feature list
manifest["feature_groups"]["weather_features"] = [
    c for c in manifest["feature_groups"]["weather_features"] if c not in dropped_features
]
manifest["feature_groups"]["all_model_features"] = ALL_FEATURES
manifest["metadata"]["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
manifest["metadata"]["dropped_features"] = dropped_features

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(manifest), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE")
print(sep)
print(f"Split config : {SPLIT_CONFIG_PATH}")
print(f"Validation   : {VALIDATION_REPORT}")
print(f"Manifest updated (visibility removed)")
print("Next: Section 3 — Notebook 06 close-out")
print(sep)

Notebook 06 | Section 2 — Validate + time split
Target : delay_in_min | Metric: MAE
Train  : 2024-07, 2024-08
Test   : 2024-09

Loading: ice_modeling_ready_2024-07.parquet
Loading: ice_modeling_ready_2024-08.parquet
Loading: ice_modeling_ready_2024-09.parquet
Total rows loaded: 377,026

FEATURE VALIDATION
  [OK] dep_hour                       nulls=0 (0.0%)
  [OK] dep_day_of_week                nulls=0 (0.0%)
  [OK] dep_month                      nulls=0 (0.0%)
  [OK] is_weekend                     nulls=0 (0.0%)
  [OK] is_rush_hour                   nulls=0 (0.0%)
  [OK] eva                            nulls=0 (0.0%)
  [OK] train_number                   nulls=0 (0.0%)
  [OK] train_line_station_num         nulls=0 (0.0%)
  [OK] final_destination_station      nulls=0 (0.0%)
  [OK] temperature_2m                 nulls=0 (0.0%)
  [OK] precipitation                  nulls=0 (0.0%)
  [OK] windspeed_10m                  nulls=0 (0.0%)
  [OK] windgusts_10m                  nulls=0 (0.0%)
  [O

# Notebook 06 — Feature Engineering
## Section 3: Close-Out

Verify feature + train/test files exist on disk and summarize Notebook 06.

**Next:** Notebook 07 — Baseline regression models (MAE).

In [4]:
# =============================================================================
# Notebook 06 | Section 3: Close-Out
# =============================================================================
# Verify feature engineering outputs; save notebook_06_summary.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
MANIFEST_PATH = REFERENCE_DIR / "feature_manifest.json"
SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
VALIDATION_REPORT = REFERENCE_DIR / "feature_validation_report.json"
FEATURE_REPORT = REFERENCE_DIR / "feature_engineering_report_multi_month.json"
SUMMARY_PATH = REFERENCE_DIR / "notebook_06_summary.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


config = load_json(CONFIG_PATH)
manifest = load_json(MANIFEST_PATH)
split_cfg = load_json(SPLIT_CONFIG_PATH)
TARGET_MONTHS = config["scope"]["target_months"]
PRIMARY_TARGET = config["ml_tasks"]["primary"]["target"]

print("Notebook 06 | Section 3 — Close-out")
print(f"Target : {PRIMARY_TARGET} | Metric: MAE")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    size = path.stat().st_size if ok else 0
    checklist.append(
        {
            "label": label,
            "path": str(path),
            "exists": ok,
            "size_mb": round(size / 1e6, 2) if ok else 0.0,
        }
    )
    status = "OK" if ok else "MISSING"
    size_txt = f"{size / 1e6:.1f} MB" if ok else "—"
    print(f"  [{status}] {label:<48} {size_txt}")
    return ok


print("=" * 72)
print("FILE CHECKLIST")
print("=" * 72)

all_ok = True

print("\nModeling-ready (Section 1)")
for month in TARGET_MONTHS:
    all_ok &= check(
        f"ice_modeling_ready_{month}.parquet",
        PROCESSED_DIR / f"ice_modeling_ready_{month}.parquet",
    )
all_ok &= check(
    "ice_modeling_ready_all_months.parquet",
    PROCESSED_DIR / "ice_modeling_ready_all_months.parquet",
)

print("\nTrain / test (Section 2)")
all_ok &= check("ice_train.parquet", TRAIN_PATH)
all_ok &= check("ice_test.parquet", TEST_PATH)

print("\nReports / config")
all_ok &= check("feature_manifest.json", MANIFEST_PATH)
all_ok &= check("train_test_split_config.json", SPLIT_CONFIG_PATH)
all_ok &= check("feature_validation_report.json", VALIDATION_REPORT)
all_ok &= check("feature_engineering_report_multi_month.json", FEATURE_REPORT)

# Row counts from disk
train_df = pd.read_parquet(TRAIN_PATH, columns=["id", PRIMARY_TARGET])
test_df = pd.read_parquet(TEST_PATH, columns=["id", PRIMARY_TARGET])

features = manifest["feature_groups"]["all_model_features"]
dropped = manifest.get("metadata", {}).get("dropped_features", split_cfg.get("dropped_features", []))

print()
print("=" * 72)
print("NOTEBOOK 06 SUMMARY")
print("=" * 72)
print(f"  Features kept     : {len(features)}")
print(f"  Features dropped  : {dropped}")
print(f"  Train rows        : {len(train_df):,}  ({', '.join(split_cfg['split']['train_months'])})")
print(f"  Test rows         : {len(test_df):,}  ({', '.join(split_cfg['split']['test_months'])})")
print(f"  Train mean delay  : {train_df[PRIMARY_TARGET].mean():.2f} min")
print(f"  Test mean delay   : {test_df[PRIMARY_TARGET].mean():.2f} min")
print()
print("  What we built:")
print("    1. Time + operational + weather features")
print("    2. Dropped visibility (100% null)")
print("    3. Time split: train Jul+Aug, test Sep")
print("    4. Saved train/test parquets on disk")
print()

ready = bool(all_ok)
print("=" * 72)
print(f"Ready for Notebook 07 (baseline models): {'YES' if ready else 'NO'}")
print("=" * 72)
if not ready:
    missing = [c["label"] for c in checklist if not c["exists"]]
    raise FileNotFoundError(f"Missing files: {missing}")

summary = {
    "metadata": {
        "notebook": "06_Feature_Engineering",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_months": TARGET_MONTHS,
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "checklist": checklist,
    "all_files_ok": ready,
    "features": {
        "count": len(features),
        "columns": features,
        "dropped": dropped,
        "operational_only": split_cfg["feature_sets_for_ablation"]["operational_only"],
        "full_with_weather": split_cfg["feature_sets_for_ablation"]["full_with_weather"],
    },
    "split": split_cfg["split"],
    "train_delay_mean": round(float(train_df[PRIMARY_TARGET].mean()), 2),
    "test_delay_mean": round(float(test_df[PRIMARY_TARGET].mean()), 2),
    "next_notebook": {
        "id": "07",
        "goal": "Baseline regression models — mean/median + Linear/Ridge; evaluate with MAE",
        "inputs": ["ice_train.parquet", "ice_test.parquet", "train_test_split_config.json"],
    },
    "ready_for_notebook_07": ready,
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print()
print(f"Saved: {SUMMARY_PATH}")
print("Next: Notebook 07 — Baseline Models (MAE)")

Notebook 06 | Section 3 — Close-out
Target : delay_in_min | Metric: MAE

FILE CHECKLIST

Modeling-ready (Section 1)
  [OK] ice_modeling_ready_2024-07.parquet               3.4 MB
  [OK] ice_modeling_ready_2024-08.parquet               3.1 MB
  [OK] ice_modeling_ready_2024-09.parquet               3.1 MB
  [OK] ice_modeling_ready_all_months.parquet            9.6 MB

Train / test (Section 2)
  [OK] ice_train.parquet                                6.5 MB
  [OK] ice_test.parquet                                 3.1 MB

Reports / config
  [OK] feature_manifest.json                            0.0 MB
  [OK] train_test_split_config.json                     0.0 MB
  [OK] feature_validation_report.json                   0.0 MB
  [OK] feature_engineering_report_multi_month.json      0.0 MB

NOTEBOOK 06 SUMMARY
  Features kept     : 15
  Features dropped  : ['visibility']
  Train rows        : 255,062  (2024-07, 2024-08)
  Test rows         : 121,964  (2024-09)
  Train mean delay  : 11.04 min
  Te